In [ ]:
import glob
import pandas as pd
import sys
from collections import Counter
import re

In [ ]:
csvs = glob.glob('parsed*')
urls = {
    'news':set(),
    'magzine':set(),
    'jobs':set(),
    'material':set(),
    'photos':set(),
    'social media':set(),
    'help':set(),
    'search':set(),
    'mobile news':set(),
    'mobile magzine':set(),
    'mobile jobs':set(),
    'mobile material':set(),
    'mobile photos':set(),
    'mobile social media':set(),
    'no cat':set()
}

In [ ]:
# for csv in sorted(csvs):
#     if '01.csv' in csv:
#         print(csv)
#     df = pd.read_csv(csv)
#     for index,row in df.iterrows():
#         if row['Content Type'] not in ['html','php']:
#             if pd.notna(row['Category']):
#                 urls[row['Category']].add(row['URL'])
#             else:
#                 urls['no cat'].add(row['URL'])

not_urls = {
    'news':set(),
    'magzine':set(),
    'jobs':set(),
    'material':set(),
    'photos':set(),
    'social media':set(),
    'help':set(),
    'search':set(),
    'mobile news':set(),
    'mobile magzine':set(),
    'mobile jobs':set(),
    'mobile material':set(),
    'mobile photos':set(),
    'mobile social media':set(),
    'no cat':set()
}
for csv in sorted(csvs):
    if '01.csv' in csv:
        print(csv)
    df = pd.read_csv(csv)
    for index,row in df.iterrows():
        if row['Content Type'] not in ['html','php']:
            if pd.notna(row['Category']):
                not_urls[row['Category']].add(row['URL'])
            else:
                not_urls['no cat'].add(row['URL'])

In [ ]:
print('URLs per Category')
total_len = 0
for key,val in urls.items():
    print(f'{key}: {len(val)}')
    total_len += len(val)
print(total_len)

In [ ]:
print('URLs per Category')
total_len = 0
for key,val in not_urls.items():
    print(f'{key}: {len(val)}')
    total_len += len(val)
print(total_len)

In [ ]:
combined = pd.read_csv('combined.csv')

In [ ]:
type(urls)  

In [ ]:
combined.head(2)

In [ ]:
def extract_idx(url):
    if 'idx=' in url:
        pattern = r'idx=(\d+)'
    elif 'id=' in url:
        pattern = r'id=(\d+)'
    match = re.search(pattern, url)
    if match:
        return match.group(1)
    return None

In [ ]:
news = combined[combined['Category'] == 'news']
condition = news['URL'].apply(lambda x :'news_id=' in x )

In [ ]:
news_links = news[condition]['URL']
news_links.str.split('.html')[news_links.str.split('.html').apply(lambda x: x[0]).apply(lambda x: 'daum' not in x)].apply(lambda x: x[0]).unique()

In [ ]:
news_urls = news_links.apply(extract_idx)
news_urls = news_urls.dropna().apply(lambda x: 'https://www.lafent.com/inews/news_view.html?news_id=' + x)
news_urls.nunique()

In [ ]:
jobs = combined[combined['Category'] == 'jobs']
condition = jobs['URL'].apply(lambda x: ('https://www.lafent.com/jobse/job_view' in x or 'https://www.lafent.com/jobse/job%5fview' in x) and 'idx=' in x)

In [ ]:
job_links = jobs[condition]['URL']

In [ ]:
job_urls = job_links.apply(extract_idx)
job_urls = job_urls.dropna().apply(lambda x: 'https://www.lafent.com/jobse/job_view?idx=' + x)
job_urls.nunique()

In [ ]:
socials = combined[combined['Category'] == 'social media']
condition = socials['URL'].apply(lambda x: ('idx=' in x and ('qna_view' in x or 'qna_%5' in x or 'freetalk' in x or 'agrbrd' in x)) or ('id=' in x and 'news' in x))
general = socials[condition]['URL'] 

array(['https://www.lafent.com/sns/news_view',
       'https://www.lafent.com/sns/qna_view',
       'https://www.lafent.com/sns/freetalk_view',
       'https://www.lafent.com/sns/agrbrd_view',
       'https://www.lafent.com/sns/agrbrd%5fview',
       'https://www.lafent.com/sns/news%5fview',
       'https://www.lafent.com/sns/freetalk%5fview'], dtype=object)

In [ ]:
qna = general[general.apply(lambda x: 'qna' in x or 'qna_%5' in x)]
free = general[general.apply(lambda x: 'freetalk' in x)]
agr = general[general.apply(lambda x: 'agrbrd' in x)]
soc_news = general[general.apply(lambda x: 'news' in x)]

In [ ]:
qna_links = qna.apply(extract_idx)
qna_links = qna_links.dropna().apply(lambda x: 'http://www.lafent.com/sns/qna_view.html?idx=' + x)
qna_links.nunique()

In [ ]:
free_links = free.apply(extract_idx)
free_links = free_links.dropna().apply(lambda x: 'http://www.lafent.com/sns/freetalk_view.html?idx=' + x)
free_links.nunique()

In [ ]:
agr_links = agr.apply(extract_idx)
agr_links = agr_links.dropna().apply(lambda x: 'http://www.lafent.com/sns/agrbrd_view.html?idx=' + x)
agr_links.nunique()

In [ ]:
soc_news_links = soc_news.apply(extract_idx)
soc_news_links = soc_news_links.dropna().apply(lambda x: 'http://www.lafent.com/sns/news_view.html?news_id=' + x)
soc_news_links.nunique()

In [ ]:
# news_urls.to_csv('news_links.csv',index=False,encoding='utf-8-sig')
# job_urls.to_csv('job.csv',index=False,encoding='utf-8-sig')
# qna_links.to_csv('qna.csv',index=False,encoding='utf-8-sig')
# free_links.to_csv('free.csv',index=False,encoding='utf-8-sig')
# agr_links.to_csv('agr.csv',index=False,encoding='utf-8-sig')
# soc_news_links.to_csv('soc_news.csv',index=False,encoding='utf-8-sig')

In [ ]:
import os
# os.remove('news.csv')
os.remove('job.csv')
os.remove('qna.csv')
os.remove('free.csv')
os.remove('agr.csv')

In [458]:
df = pd.read_csv('news.csv')
df_cleaned = df.dropna(subset=['title'])
df_cleaned.to_csv('final_news.csv',index=False,encoding='utf-8-sig')

In [318]:
# df = pd.read_csv('jobs.csv')
# 1698 - (len(df) - 4466)
# # df.drop_duplicates(['company','대표자','사원수','주요분야','자본금','주요업종','매출액','소재지역','홈페이지','담당업무','근무지역','고용형태','근무부서','복리후생','급여조건','모집인원','경력사항','나이제한','최종학력','우대사항','접수기간','상세설명','keywords'])
# df_cleaned = df.dropna(subset=['company'])
# df_cleaned.to_csv('final_jobs.csv',index=False,encoding='utf-8-sig')
df = pd.read_csv('final_jobs.csv')
df

,company,대표자,주요분야,주요업종,소재지역,사원수,자본금,매출액,홈페이지,담당업무,...,급여조건,모집인원,경력사항,나이제한,최종학력,우대사항,접수기간,상세설명,keywords,href
0,(주)희림종합건축사사무소,정영균,계획·설계·디자인,조경·건설공통,서울,"1,320",70억,"2,039억",www.heerim.com,"계획·설계·디자인>기본 및 실시설계,디자인",...,회사내규에 따름,0,경력(6년이상 ~ 8년미만),무관,전문학사 이상,"디자인, 스케치, 모델링 및 랜더링, BIM",2023년 07월 21일 (금) ~ 2023년 08월 31일 (목),"2023 희림종합건축사사무소 조경설계팀 경력직 모집\n건축, 도시의 외부공간 및 그...","['설계', '실시설계', '기본설계', '건축조경', '조경', '디자인', '계획']",https://www.lafent.com/jobse/job_view?idx=24774
1,제이엘이티 디자인그룹,"정동진, 정일태",계획·설계·디자인,설계사무소,서울,6,NaN,NaN,www.jlet.co.kr,계획·설계·디자인>계획 및 설계,...,회사내규에 따름,1,경력(1년이상 ~ 10년미만),무관,전문학사 이상,관련학과 졸업자,2023년 08월 24일 (목) ~ 2023년 09월 30일 (토),2019년 설립된 제이엘이티 디자인그룹은 젊은 조경가들이 함께 성장하는 회사입니다....,"['조경', '설계']",https://www.lafent.com/jobse/job_view?idx=24338
2,(주)수성엔지니어링,박미례,계획·설계·디자인,엔지니어링,서울,850,5,1500,www.soosungeng.com/ko/,계획·설계·디자인>조경 설계,...,회사내규에 따름,2,경력(2년이상 ~ 15년미만),무관,학사 이상,"조경학과, 조경기사, 각종 프로그램 능통자",2022년 09월 16일 (금) ~ 2022년 10월 07일 (금),회사에 있는 동안만큼은 어느 가족보다 화목한 분위기임을 자부하는\n(주) 수성엔지니...,"['조경설계', '엔지니어링']",https://www.lafent.com/jobse/job_view?idx=24874
3,디자인오키즘,김옥경,계획·설계·디자인,조경·건설공통,서울,NaN,NaN,NaN,www.okism.co.kr,계획·설계·디자인>정원조경디자인,...,회사내규에 따름,1,"신입, 경력(2년이상 ~ 3년미만)",무관,전문학사 이상,CAD 및 포토샵 능통자,2016년 02월 12일 (금) ~ 2021년 03월 01일 (월),●업무내용\n-담당업무: 정원/조경계획 및 설계\n\n●모집인원 및 지원자격\n-모...,"['조경디자인', '조경', '정원조경', '경력']",https://www.lafent.com/jobse/job_view?idx=18100
4,(주)엔에스프리,김지환,시공·공무·감리,전문건설업,경기,7,4억,NaN,NaN,시공·공무·감리>조경시공 업무 및 공무보조,...,회사내규에 따름,00,"신입, 경력(3년이상 ~ 15년미만)",무관,고등학교 졸업 이상,CAD 및 문서작성 가능자,2017년 09월 01일 (금) ~ 2022년 09월 30일 (금),"*자격요건\n-중간관리자\n-학력무관, 숙식제공,\n-우대 : 현장시공 및 공무 업...","['조경', '조경산업기사', '조경기사', '조경식재', '조경시설물', '아르바...",https://www.lafent.com/jobse/job_view?idx=20302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3615,(주)해동기술개발공사,구용호,NaN,엔지니어링,대구,40,1,NaN,NaN,>조경계획 및 설계,...,회사내규에 따름,0,"신입, 경력(이상 ~ 미만)",무관,학사 이상,NaN,2012년 06월 19일 (화) ~ 2012년 08월 20일 (월),"전형방법 : 서류전형 제출서류 : 이력서, 자기소개서, 주민등록등본",[],https://www.lafent.com/jobse/job_view?idx=13380
3616,(주)도시녹화조경,유은희,NaN,시공·관리·감리공통,충남,5,2,10억,NaN,">소장,공무,관리,경리",...,회사내규에 따름,0,경력(2년이상 ~ 미만),22세 이상 ~ 35세 미만,전문학사 이상,NaN,2008년 10월 13일 (월) ~ 2008년 11월 29일 (토),회사는 대전이고 현장은 충청권입니다. 많은 지원바랍니다. 이메일로 이력서 보내주세요^^,[],https://www.lafent.com/jobse/job_view?idx=6220
3617,(주)도시녹화조경,유은희,NaN,시공·관리·감리공통,대전,5,2,10억,NaN,">현장 소장, 대리인, 기사",...,회사내규에 따름,2,"신입, 경력(이상 ~ 미만)",25세 이상 ~ 35세 미만,전문학사 이상,NaN,2009년 02월 02일 (월) ~ 2009년 02월 28일 (토),"1. 공사부(현장관리인) : 현장소장,대리인,기사 2. 자격요건 : 관련자격증(기사...",[],https://www.lafent.com/jobse/job_view?idx=6768
3618,(주)희담,안성만,시공·공무·감리,조경·건설공통,서울,7,4,57억,NaN,"시공·공무·감리>공무, 내역",...,회사내규에 따름,1,경력(3년이상 ~ 5년미만),25세 이상 ~ 33세 미만,전문학사 이상,조경기사 또는 중급자격,2019년 12월 02일 (월) ~ 2020년 02월 15일 (토),"조경공사업 및 조경식재, 조경시설물설치공사업 면허를 보유하고 있으며\n서울시 관공사...","['조경', '공무', '설계변경', '시공']",https://www.lafent.com/jobse/job_view?idx=22547
